## Preprocessing - Version 1 (Maelle)

Ce notebook construit une **première version propre et reproductible du preprocessing** pour ton projet de Machine Learning 2.

Objectifs principaux :
- **Charger** les données (mêmes chemins / colonnes que le notebook starter).
- **Nettoyer / harmoniser** les réponses catégorielles (mêmes réponses écrites différemment).
- **Encoder** les variables catégorielles.
- **Gérer les valeurs manquantes** et les colonnes très corrélées.
- **Traiter simplement les colonnes texte** (ou au moins préparer le terrain).

In [12]:
# Imports principaux
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.base import BaseEstimator, TransformerMixin

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

### 1. Chargement des données (en s'inspirant du notebook starter)

Ton professeur fournit dans `Starter_project/02_preprocess_starter.ipynb` un exemple très simple qui :
- charge `../data/train.csv` et `../data/test.csv`
- utilise `target_is_fraud` comme cible et `customer_id` comme identifiant
- applique un preprocessing **numérique uniquement**.

Ici on garde les **mêmes chemins et les mêmes noms de colonnes** par défaut pour rester cohérent avec le starter, mais on va construire un preprocessing plus riche (texte, catégories, corrélation, etc.).

Tu peux bien sûr adapter les chemins / noms de colonnes si votre projet a évolué.

In [13]:
# Paramètres par défaut alignés sur le notebook starter
TRAIN_PATH = "../data/kaggle_b2_fraud_train_v4.csv"
TEST_PATH = "../data/kaggle_b2_fraud_test_v4.csv"
TARGET_COL = "target_is_fraud"   # cible utilisée dans le starter
ID_COL = "customer_id"           # identifiant client dans le starter

try:
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    print("train shape :", train.shape)
    print("test shape  :", test.shape)
except FileNotFoundError as e:
    print(f"Erreur de chargement : {e}")
    train = pd.DataFrame()
    test = pd.DataFrame()

df = train.copy()
print("Shape du dataframe (train) utilisé pour l'EDA :", df.shape)
df.head()

train shape : (160000, 34)
test shape  : (40000, 33)
Shape du dataframe (train) utilisé pour l'EDA : (160000, 34)


,customer_id,account_id,age,tenure_months,annual_income_eur,credit_score,num_transactions_30d,avg_amount_30d_eur,max_amount_30d_eur,days_since_last_login,support_tickets_90d,chargebacks_12m,failed_payments_6m,device_trust_z,ip_risk_z,is_vpn,num_devices_30d,is_new_device,country,region,referrer_code,signup_date,secondary_email,internal_signal_1,internal_signal_2,internal_signal_3,internal_signal_4,internal_signal_5,internal_signal_6,internal_signal_7,internal_signal_8,terms_accepted_flag,partner_risk_indicator,target_is_fraud
0,CUST_6O9Q8D4I36,ACC_TXXXTNEUVKFY,34,108,38635.01,544,20,60.92,80.16,4.9,1,0,1,-0.934,-0.355,0,1,0,CH,NaN,hvty1wydg7,2025-04-21,xor2cp@mail.com,-1.00827,2.77585,-0.35668,0.73256,-0.90929,0.38747,-0.21200,1.78510,1,NaN,0
1,CUST_FGUGTW230C,ACC_70VD7A4FFWCW,48,2,19912.97,703,21,112.11,571.12,0.3,0,0,0,-1.176,1.703,0,1,0,FR,Bretagne,pgriqsqesm,2024-03-06,10t4qa@mail.com,0.48813,-0.65897,0.50084,0.23241,-1.16817,-0.82509,-0.78065,-1.25603,1,2.244073,0
2,CUST_8ZI3LCBZ0W,ACC_AF53381QSC0L,27,0,20326.87,720,25,73.61,492.57,4.6,1,0,0,-0.482,1.020,0,1,0,CA,NaN,3je9oc60zl,2025-07-22,NaN,0.47065,0.68621,-1.30771,-0.91109,0.92254,-0.63157,0.89172,-0.34545,1,NaN,1
3,CUST_5MP3AR41CJ,ACC_U7WZGJ486LIV,45,49,38452.47,703,17,47.53,204.18,25.3,1,0,1,1.298,0.685,0,1,0,BE,NaN,hpxga58dbn,2024-02-28,NaN,-1.13048,-1.95970,-1.68684,0.85302,1.28198,1.06118,0.99292,-0.05549,1,NaN,0
4,CUST_GNPL83JB0J,ACC_XW7DS3ED5J4Y,37,46,23065.56,594,13,99.95,734.09,12.8,0,1,0,0.317,-1.930,0,1,0,FR,Nouvelle-Aquitaine,1ikbb88w6r,2025-12-22,NaN,-0.87503,0.32109,-1.51914,-0.27024,-0.61402,1.32524,0.78992,-0.96780,1,NaN,0


### 1.5. Suppression des colonnes inutiles (data leakage, redondantes, non prédictives)

Cette section supprime les colonnes qui ne doivent **PAS** être utilisées pour prédire la fraude :

**1. Data Leakage** (informations disponibles seulement après détection) :
- `chargebacks_12m`, `chargeback_resolution_time_days`, `manual_review_result`, `post_event_status_code`

**2. Identifiants purs** (pas prédictifs, gardés seulement pour export) :
- `customer_id`, `account_id`, `referrer_code`

**3. Colonnes redondantes/dérivées** (peuvent être recalculées) :
- `annual_income_log` (dérivé de `annual_income_eur`)
- `income_dup_eur` (doublon de `annual_income_eur`)
- `credit_score_scaled` (dérivé de `credit_score`)
- `tx_amount_total_30d_eur` (peut être calculé)
- `max_to_avg_ratio` (peut être calculé)

**4. Colonnes texte non exploitables** (nécessitent NLP avancé) :
- `last_ticket_subject`, `customer_note`, `secondary_email`

**5. Colonnes avec trop de valeurs manquantes** (>95% manquants) :
- `partner_risk_indicator`, `legacy_partner_score`

**6. Date brute** (sera transformée en features si nécessaire) :
- `signup_date` (peut être convertie en features temporelles plus tard si besoin)

In [14]:
# Liste complète des colonnes à supprimer (par catégorie)

# 1. Data Leakage
COLS_DATA_LEAKAGE = [
    "chargebacks_12m",
    "chargeback_resolution_time_days",
    "manual_review_result",
    "post_event_status_code",
]

# 2. Identifiants purs (à garder pour export mais pas pour le modèle)
# Note: customer_id sera gardé pour l'export final, mais retiré avant preprocessing
COLS_IDENTIFIERS = [
    "account_id",
    "referrer_code",
]

# 3. Colonnes redondantes/dérivées
COLS_REDUNDANT = [
    "annual_income_log",      # dérivé de annual_income_eur
    "income_dup_eur",         # doublon de annual_income_eur
    "credit_score_scaled",    # dérivé de credit_score (on scalera dans le preprocessing)
    "tx_amount_total_30d_eur", # peut être calculé à partir de num_transactions_30d * avg_amount_30d_eur
    "max_to_avg_ratio",       # peut être calculé à partir de max_amount_30d_eur / avg_amount_30d_eur
]

# 4. Colonnes texte non exploitables (sauf si traitement NLP avancé)
COLS_TEXT_UNUSABLE = [
    "last_ticket_subject",
    "customer_note",
    "secondary_email",  # aussi très peu rempli (12609/160000)
]

# 5. Colonnes avec trop de valeurs manquantes (>95%)
COLS_TOO_MANY_MISSING = [
    "partner_risk_indicator",  # seulement 4751/160000 non-null (~3%)
    "legacy_partner_score",   # seulement 6264/160000 non-null (~4%)
]

# 6. Date brute (peut être transformée en features temporelles plus tard si besoin)
COLS_DATE_RAW = [
    "signup_date",  # format string, peut être convertie en features temporelles si nécessaire
]

# Liste finale de toutes les colonnes à supprimer
COLS_TO_DROP = (
    COLS_DATA_LEAKAGE +
    COLS_IDENTIFIERS +
    COLS_REDUNDANT +
    COLS_TEXT_UNUSABLE +
    COLS_TOO_MANY_MISSING +
    COLS_DATE_RAW
)

print(f"Total de colonnes à supprimer : {len(COLS_TO_DROP)}")
print("\nPar catégorie :")
print(f"  - Data Leakage: {len(COLS_DATA_LEAKAGE)}")
print(f"  - Identifiants: {len(COLS_IDENTIFIERS)}")
print(f"  - Redondantes: {len(COLS_REDUNDANT)}")
print(f"  - Texte non exploitable: {len(COLS_TEXT_UNUSABLE)}")
print(f"  - Trop de valeurs manquantes: {len(COLS_TOO_MANY_MISSING)}")
print(f"  - Date brute: {len(COLS_DATE_RAW)}")

# Suppression sur train et test
if not train.empty:
    cols_exist_train = [col for col in COLS_TO_DROP if col in train.columns]
    cols_exist_test = [col for col in COLS_TO_DROP if col in test.columns]
    
    if cols_exist_train:
        train = train.drop(columns=cols_exist_train)
        print(f"\n✅ {len(cols_exist_train)} colonnes supprimées du train :")
        for col in cols_exist_train:
            print(f"   - {col}")
    else:
        print("\n⚠️ Aucune colonne à supprimer trouvée dans train")
    
    if cols_exist_test:
        test = test.drop(columns=cols_exist_test)
        print(f"\n✅ {len(cols_exist_test)} colonnes supprimées du test")
    
    # Mise à jour de df (utilisé pour l'EDA)
    df = train.copy()
    print(f"\n📊 Nouvelle shape train : {train.shape}")
    print(f"📊 Nouvelle shape test  : {test.shape}")
    print(f"📊 Colonnes restantes : {train.shape[1]}")
else:
    print("⚠️ train est vide, impossible de supprimer les colonnes")

Total de colonnes à supprimer : 17

Par catégorie :
  - Data Leakage: 4
  - Identifiants: 2
  - Redondantes: 5
  - Texte non exploitable: 3
  - Trop de valeurs manquantes: 2
  - Date brute: 1

✅ 6 colonnes supprimées du train :
   - chargebacks_12m
   - account_id
   - referrer_code
   - secondary_email
   - partner_risk_indicator
   - signup_date

✅ 6 colonnes supprimées du test

📊 Nouvelle shape train : (160000, 28)
📊 Nouvelle shape test  : (40000, 27)
📊 Colonnes restantes : 28


### 2. Vue d'ensemble rapide (rappel EDA)

On affiche quelques infos générales pour vérifier que tout est cohérent avec ce que tu as vu en EDA :
- types de colonnes
- nombre de valeurs manquantes
- exemples de valeurs uniques sur quelques colonnes catégorielles

Tu peux adapter la liste `sample_cat_cols` selon les variables importantes de votre projet.

In [15]:
if not df.empty:
    display(df.info())

    # Valeurs manquantes par colonne
    missing = df.isna().mean().sort_values(ascending=False)
    print("\nTaux de valeurs manquantes par colonne (top 20) :")
    display(missing.head(20))

    # Exemple de colonnes catégorielles à inspecter (à adapter)
    sample_cat_cols = [col for col in df.columns if df[col].dtype == "object"][:10]
    print("\nExemples de valeurs uniques pour quelques colonnes catégorielles :")
    for col in sample_cat_cols:
        print(f"\nColonne: {col}")
        print(df[col].value_counts(dropna=False).head(20))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160000 entries, 0 to 159999
Data columns (total 28 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   customer_id            160000 non-null  object 
 1   age                    160000 non-null  int64  
 2   tenure_months          160000 non-null  int64  
 3   annual_income_eur      160000 non-null  float64
 4   credit_score           160000 non-null  int64  
 5   num_transactions_30d   160000 non-null  int64  
 6   avg_amount_30d_eur     160000 non-null  float64
 7   max_amount_30d_eur     160000 non-null  float64
 8   days_since_last_login  160000 non-null  float64
 9   support_tickets_90d    160000 non-null  int64  
 10  failed_payments_6m     160000 non-null  int64  
 11  device_trust_z         160000 non-null  float64
 12  ip_risk_z              160000 non-null  float64
 13  is_vpn                 160000 non-null  int64  
 14  num_devices_30d        160000 non-nu

None


Taux de valeurs manquantes par colonne (top 20) :


region                 0.2831
customer_id            0.0000
age                    0.0000
terms_accepted_flag    0.0000
internal_signal_8      0.0000
internal_signal_7      0.0000
internal_signal_6      0.0000
internal_signal_5      0.0000
internal_signal_4      0.0000
internal_signal_3      0.0000
internal_signal_2      0.0000
internal_signal_1      0.0000
country                0.0000
is_new_device          0.0000
num_devices_30d        0.0000
is_vpn                 0.0000
ip_risk_z              0.0000
device_trust_z         0.0000
failed_payments_6m     0.0000
support_tickets_90d    0.0000
dtype: float64


Exemples de valeurs uniques pour quelques colonnes catégorielles :

Colonne: customer_id
customer_id
CUST_6O9Q8D4I36    1
CUST_5JTNWOQ33U    1
CUST_351S1SNSKF    1
CUST_2J3L4DIX4Q    1
CUST_8OJ6DCPB7A    1
CUST_WE2WIXTB5X    1
CUST_PELFCWDIGQ    1
CUST_VO84MX2M54    1
CUST_XXYOB5AAB6    1
CUST_NUN46ZBDNA    1
CUST_AWU01LRJNA    1
CUST_MO72BPGP8J    1
CUST_LBLLQ2UYJ9    1
CUST_95H3TGVK02    1
CUST_6DN4BZCWV5    1
CUST_5A4PUWDCK6    1
CUST_PI3KB05CCC    1
CUST_VZ8CZO8OUG    1
CUST_WVAB2XG8HZ    1
CUST_L6CW4EX9SR    1
Name: count, dtype: int64

Colonne: country
country
FR    114704
CH      8001
BE      7802
DE      6314
IT      4849
ES      4752
GB      4733
NL      3261
CA      1653
US      1588
MA       798
TN       781
DZ       764
Name: count, dtype: int64

Colonne: region
region
NaN                           45296
Île-de-France                 32032
Auvergne-Rhône-Alpes          14985
Provence-Alpes-Côte d'Azur     9272
Nouvelle-Aquitaine             9182
Hauts-de-France            

### 4. Séparation variables numériques / catégorielles / texte

On va séparer :
- **numériques**: `int`, `float`
- **catégorielles**: `object`, mais avec peu de modalités
- **texte libre**: `object` avec beaucoup de modalités uniques (par ex. des descriptions longues)

Pour l'instant, on va :
- **garder les colonnes texte libre, mais ne pas les encoder tout de suite**, ou
- optionnellement créer une version très simple (longueur du texte) si tu veux les utiliser vite fait dans un modèle linéaire/arbre.

Tu pourras ensuite affiner avec TF-IDF, embeddings, etc., si nécessaire.

In [16]:
if not df.empty:
    num_cols = df.select_dtypes(include=["number"]).columns.tolist()
    obj_cols = df.select_dtypes(include=["object"]).columns.tolist()

    # Heuristique simple : colonnes objet avec beaucoup de valeurs uniques = texte libre
    text_cols = []
    cat_cols = []
    for col in obj_cols:
        n_unique = df[col].nunique(dropna=True)  # CORRECTION : utiliser df au lieu de df_norm
        if n_unique > 30:  # seuil à adapter selon ton dataset
            text_cols.append(col)
        else:
            cat_cols.append(col)

    print("Colonnes numériques (example 10) :", num_cols[:10])
    print("Colonnes catégorielles (example 10) :", cat_cols[:10])
    print("Colonnes texte libre (example 10) :", text_cols[:10])
else:
    num_cols, cat_cols, text_cols = [], [], []

Colonnes numériques (example 10) : ['age', 'tenure_months', 'annual_income_eur', 'credit_score', 'num_transactions_30d', 'avg_amount_30d_eur', 'max_amount_30d_eur', 'days_since_last_login', 'support_tickets_90d', 'failed_payments_6m']
Colonnes catégorielles (example 10) : ['country', 'region']
Colonnes texte libre (example 10) : ['customer_id']


### 5. Pipeline de preprocessing (imputation, encodage, scaling)

Ici on construit un **ColumnTransformer** scikit-learn qui applique :
- aux **numériques** : imputation (médiane) + standardisation
- aux **catégorielles** : imputation (valeur la plus fréquente) + One-Hot Encoding
- aux **textes** (optionnel) : on peut par exemple ne garder que la **longueur du texte** comme feature simple.

On ajoute aussi une étape pour **supprimer les features très peu variables** (VarianceThreshold), ce qui aide parfois à nettoyer un peu.

In [17]:
class TextLengthExtractor(BaseEstimator, TransformerMixin):
    """Transforme des colonnes texte en une seule feature : longueur du texte."""

    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # Si une seule colonne texte, on renvoie simplement la longueur
        if isinstance(X, pd.DataFrame):
            return X.apply(lambda col: col.astype(str).str.len(), axis=0)
        else:
            return pd.DataFrame(X).astype(str).apply(lambda col: col.str.len(), axis=0)


# NOTE: Cette section est optionnelle/test pour voir le pipeline sur df
# La version finale et complète est dans la section 8 qui utilise train/test directement
if not df.empty:
    # Si tu as une colonne cible, on la sépare
    if TARGET_COL is not None and TARGET_COL in df.columns:
        y = df[TARGET_COL]
        X = df.drop(columns=[TARGET_COL])
    else:
        y = None
        X = df.copy()
    
    # Retirer aussi customer_id s'il est présent (identifiant, pas une feature)
    if ID_COL in X.columns:
        X = X.drop(columns=[ID_COL])

    # IMPORTANT : Recalculer num_cols, cat_cols, text_cols sur X (pas sur df)
    # car X n'a plus target_is_fraud ni customer_id
    num_cols_X = X.select_dtypes(include=["number"]).columns.tolist()
    obj_cols_X = X.select_dtypes(include=["object"]).columns.tolist()

    text_cols_X = []
    cat_cols_X = []
    for col in obj_cols_X:
        n_unique = X[col].nunique(dropna=True)
        if n_unique > 30:  # seuil à adapter selon ton dataset
            text_cols_X.append(col)
        else:
            cat_cols_X.append(col)

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    text_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="")),
        ("length", TextLengthExtractor()),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_cols_X),
            ("cat", categorical_transformer, cat_cols_X),
            ("text", text_transformer, text_cols_X),
        ],
        remainder="drop",  # on ne garde pas les colonnes non listées
    )

    full_pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("var_thresh", VarianceThreshold(threshold=0.0)),  # supprime colonnes constantes
    ])

    X_prep = full_pipeline.fit_transform(X)
    print("Shape de X avant preprocessing :", X.shape)
    print("Shape de X après preprocessing :", X_prep.shape)
else:
    print("⚠️ df est vide, impossible de construire le pipeline pour l'instant.")

Shape de X avant preprocessing : (160000, 26)
Shape de X après preprocessing : (160000, 48)


### 6. Suppression des colonnes très corrélées

**⚠️ NOTE : Cette section est optionnelle/test. La version finale complète est dans la section 8.**

Tu voulais **supprimer les colonnes qui ont un taux de corrélation très très élevé**.

Pour ça, on peut travailler au niveau des **variables numériques brutes** (avant encodage) :
- on calcule la matrice de corrélation sur les colonnes numériques
- pour chaque paire de variables avec corrélation `|corr| > seuil` (par défaut 0.95), on supprime l'une des deux (par exemple la seconde dans la paire).

Ça reprendra en partie ce que vous aviez sûrement vu en EDA (corrélogramme, etc.). Tu peux adapter le seuil si besoin.

In [18]:
def drop_highly_correlated_features(df_in, threshold=0.95, cols_to_exclude=None):
    
    if cols_to_exclude is None:
        cols_to_exclude = []

    df_num = df_in.select_dtypes(include=["number"]).copy()
    corr_matrix = df_num.corr().abs()

    # Triangulaire supérieure pour ne pas dupliquer les paires
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    to_drop = set()
    for col in upper.columns:
        if col in cols_to_exclude:
            continue
        # si une autre colonne est très corrélée à col, on marque col à supprimer
        if any(upper[col] > threshold):
            to_drop.add(col)

    print(f"Nombre de colonnes numériques avant filtrage: {df_num.shape[1]}")
    print(f"Colonnes marquées à supprimer (corr > {threshold}): {len(to_drop)}")

    df_out = df_in.drop(columns=list(to_drop))
    return df_out, list(to_drop)


# NOTE: Cette section teste la fonction sur df (optionnel)
# La version finale utilise cette fonction dans la section 8 sur train_norm
if not df.empty:
    df_corr_filtered, dropped_cols = drop_highly_correlated_features(df, threshold=0.95)
    print("Colonnes supprimées pour forte corrélation:")
    print(dropped_cols)
    print("Shape avant / après filtrage corrélations:", df.shape, "->", df_corr_filtered.shape)
else:
    df_corr_filtered, dropped_cols = pd.DataFrame(), []

Nombre de colonnes numériques avant filtrage: 25
Colonnes marquées à supprimer (corr > 0.95): 0
Colonnes supprimées pour forte corrélation:
[]
Shape avant / après filtrage corrélations: (160000, 28) -> (160000, 28)


### 7. Pipeline complet réappliqué après filtrage de corrélation (OPTIONNEL/TEST)

**⚠️ NOTE : Cette section est optionnelle/test pour voir le pipeline sur `df`. La version finale complète est dans la section 8 qui utilise `train` et `test` directement.**

On reconstruit `X` à partir de `df_corr_filtered` et on ré-applique le pipeline de preprocessing.

C'est ce **X final preprocessé** (et éventuellement `y`) que tu pourras ensuite utiliser pour l'entraînement de tes modèles (RandomForest, XGBoost, etc.).

In [19]:
if not df_corr_filtered.empty:
    # Re-définition X / y après filtrage des colonnes corrélées
    if TARGET_COL is not None and TARGET_COL in df_corr_filtered.columns:
        y_final = df_corr_filtered[TARGET_COL]
        X_final = df_corr_filtered.drop(columns=[TARGET_COL])
    else:
        y_final = None
        X_final = df_corr_filtered

    # On recalcule les listes de colonnes
    num_cols_final = X_final.select_dtypes(include=["number"]).columns.tolist()
    obj_cols_final = X_final.select_dtypes(include=["object"]).columns.tolist()

    text_cols_final = []
    cat_cols_final = []
    for col in obj_cols_final:
        n_unique = X_final[col].nunique(dropna=True)
        if n_unique > 30:
            text_cols_final.append(col)
        else:
            cat_cols_final.append(col)

    numeric_transformer_final = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer_final = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    text_transformer_final = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="")),
        ("length", TextLengthExtractor()),
    ])

    preprocessor_final = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer_final, num_cols_final),
            ("cat", categorical_transformer_final, cat_cols_final),
            ("text", text_transformer_final, text_cols_final),
        ],
        remainder="drop",
    )

    full_pipeline_final = Pipeline(steps=[
        ("preprocessor", preprocessor_final),
        ("var_thresh", VarianceThreshold(threshold=0.0)),
    ])

    X_preprocessed = full_pipeline_final.fit_transform(X_final)

    print("Shape X_final avant preprocessing :", X_final.shape)
    print("Shape X_preprocessed après preprocessing :", X_preprocessed.shape)

    # Optionnel : train / test split si tu veux tout préparer ici
    if y_final is not None:
        X_train, X_test, y_train, y_test = train_test_split(
            X_preprocessed,
            y_final,
            test_size=0.2,
            random_state=42,
            stratify=y_final if y_final.nunique() < 20 else None,
        )
        print("X_train:", X_train.shape, "X_test:", X_test.shape)
else:
    print("⚠️ df_corr_filtered est vide, vérifie le chargement des données.")

Shape X_final avant preprocessing : (160000, 27)
Shape X_preprocessed après preprocessing : (160000, 48)
X_train: (128000, 48) X_test: (32000, 48)


### 7.5. Définition des classes utilitaires (TextNormalizer, TextLengthExtractor)

Ces classes sont nécessaires pour la section 8. Elles permettent de normaliser les textes et d'extraire des features simples des colonnes texte.

In [20]:
class TextNormalizer(BaseEstimator, TransformerMixin):
    """Nettoie les colonnes de type texte/catégorielles.

    - met en minuscule
    - strip espaces début/fin
    - remplace quelques variantes fréquentes (oui/non, etc.)
    - applique des mappings personnalisés par colonne
    """

    def __init__(self, columns=None, custom_mappings=None):
        self.columns = columns
        # custom_mappings est un dict du type {"nom_colonne": {"valeur_brute": "valeur_normalisée"}}
        self.custom_mappings = custom_mappings if custom_mappings is not None else {}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        cols = self.columns if self.columns is not None else X.select_dtypes(include=["object"]).columns

        global_map = {
            "oui": "oui",
            "yes": "oui",
            "y": "oui",
            "o": "oui",
            "non": "non",
            "no": "non",
            "n": "non",
        }

        for col in cols:
            if col not in X.columns:
                continue

            X[col] = (
                X[col]
                .astype(str)
                .str.strip()
                .str.lower()
                .replace("nan", np.nan)
            )

            # Mapping global générique (oui/non, etc.)
            X[col] = X[col].replace(global_map)

            # Mapping spécifique à la colonne si défini
            if col in self.custom_mappings:
                X[col] = X[col].replace(self.custom_mappings[col])

        return X


class TextLengthExtractor(BaseEstimator, TransformerMixin):
    """Transforme des colonnes texte en une seule feature : longueur du texte."""

    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # Si une seule colonne texte, on renvoie simplement la longueur
        if isinstance(X, pd.DataFrame):
            return X.apply(lambda col: col.astype(str).str.len(), axis=0)
        else:
            return pd.DataFrame(X).astype(str).apply(lambda col: col.str.len(), axis=0)


print("✅ Classes TextNormalizer et TextLengthExtractor définies")

✅ Classes TextNormalizer et TextLengthExtractor définies


### 8. Harmonisation explicite des colonnes `payment_method`, `occupation`, `region`

Dans cette section, on applique **explicitement** les regroupements que tu as identifiés dans Excel :
- `payment_method` :
  - `apple_pay` et `ApplePay` → même catégorie
  - `google_pay` et `google pay` → même catégorie
  - `pay pal` et `paypal` → même catégorie
- `occupation` : idem si tu as des variantes d’écriture pour un même métier
- `region` : idem pour les différentes écritures d’une même région.

On crée un `custom_mappings_example` complet, on réapplique le `TextNormalizer` à `train` **et** à `test`, puis on relance le pipeline final et on prépare des matrices prêtes pour les modèles et l’export.

In [21]:
# ⚠️ À ADAPTER : complète les mappings pour occupation et region

custom_mappings_example = {
    "payment_method": {
        # après normalisation, "ApplePay" devient "applepay"
        "apple_pay": "apple_pay",
        "applepay": "apple_pay",
        # "google pay" reste "google pay" (espaces et minuscules)
        "google_pay": "google_pay",
        "google pay": "google_pay",
        # regroupement des variantes de PayPal
        "paypal": "paypal",
        "pay pal": "paypal",
    },
   
    "occupation": {
         "free lancer": "freelancer",
         "freelancer": "freelancer",
         "public_sector": "public-sector",
         "public-sector": "public-sector",
         "self_employed": "self employed",
         "self employed": "self employed",
    },

    "region": {
         "Auvergne Rhone Alpes" : "Auvergne-Rhône-Alpes",
         "Auvergne-Rhône-Alpes" : "Auvergne-Rhône-Alpes",
         "Hauts de France" : "Hauts-de-France",
         "Hauts-de-France" : "Hauts-de-France",
         "Ile-de-France": "Ile-de-France",
         "Île-de-France": "Ile-de-France",
         "Provence Alpes Cote d Azur": "Provence-Alpes-Côte d'Azur",
         "Provence-Alpes-Côte d'Azur": "Provence-Alpes-Côte d'Azur",
     },
}

# 1) Ré-appliquer la normalisation texte sur train et test avec ces mappings
if not train.empty:
    text_norm_final = TextNormalizer(custom_mappings=custom_mappings_example)
    train_norm = text_norm_final.fit_transform(train)
    test_norm = text_norm_final.transform(test)  # IMPORTANT : transform seulement sur test

    # 2) Re-filtrage des colonnes fortement corrélées sur train uniquement
    df_corr_filtered_final, dropped_cols_final = drop_highly_correlated_features(train_norm, threshold=0.95)

    print("Colonnes supprimées pour forte corrélation (version finale):")
    print(dropped_cols_final)

    # 3) Redéfinition X / y pour train + alignement des colonnes sur test
    # IMPORTANT : retirer customer_id et target avant preprocessing (mais les garder pour export)
    
    # Sauvegarder les identifiants et la cible avant de les retirer
    if ID_COL in df_corr_filtered_final.columns:
        customer_ids_train = df_corr_filtered_final[ID_COL].copy()
    else:
        customer_ids_train = None
    
    if TARGET_COL in df_corr_filtered_final.columns:
        y_final = df_corr_filtered_final[TARGET_COL].copy()
    else:
        y_final = None
    
    # Retirer ID_COL et TARGET_COL avant preprocessing
    cols_to_drop = []
    if ID_COL in df_corr_filtered_final.columns:
        cols_to_drop.append(ID_COL)
    if TARGET_COL in df_corr_filtered_final.columns:
        cols_to_drop.append(TARGET_COL)
    
    X_final_train = df_corr_filtered_final.drop(columns=cols_to_drop)

    # Appliquer les mêmes colonnes au test (garder seulement les colonnes de X_final_train)
    cols_to_keep = X_final_train.columns.tolist()
    X_final_test = test_norm[cols_to_keep].copy()
    
    # Sauvegarder aussi les customer_id du test pour l'export
    if ID_COL in test_norm.columns:
        customer_ids_test = test_norm[ID_COL].copy()
    else:
        customer_ids_test = None

    # 4) Recalcul des listes de colonnes
    num_cols_final = X_final_train.select_dtypes(include=["number"]).columns.tolist()
    obj_cols_final = X_final_train.select_dtypes(include=["object"]).columns.tolist()

    text_cols_final = []
    cat_cols_final = []
    for col in obj_cols_final:
        n_unique = X_final_train[col].nunique(dropna=True)
        if n_unique > 30:
            text_cols_final.append(col)
        else:
            cat_cols_final.append(col)

    numeric_transformer_final = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer_final = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    text_transformer_final = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="")),
        ("length", TextLengthExtractor()),
    ])

    preprocessor_final = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer_final, num_cols_final),
            ("cat", categorical_transformer_final, cat_cols_final),
            ("text", text_transformer_final, text_cols_final),
        ],
        remainder="drop",
    )

    full_pipeline_final = Pipeline(steps=[
        ("preprocessor", preprocessor_final),
        ("var_thresh", VarianceThreshold(threshold=0.0)),
    ])

    # 5) Fit sur train, transform sur train et test
    X_train_preprocessed = full_pipeline_final.fit_transform(X_final_train)
    X_test_preprocessed = full_pipeline_final.transform(X_final_test)

    print("Shape X_final_train avant preprocessing :", X_final_train.shape)
    print("Shape X_train_preprocessed :", X_train_preprocessed.shape)
    print("Shape X_final_test avant preprocessing :", X_final_test.shape)
    print("Shape X_test_preprocessed :", X_test_preprocessed.shape)

    # 6) Option : split train/validation
    if y_final is not None:
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_train_preprocessed,
            y_final,
            test_size=0.2,
            random_state=42,
            stratify=y_final if y_final.nunique() < 20 else None,
        )
        print("X_train:", X_train.shape, "X_valid:", X_valid.shape)

    # 7) Option : préparer les DataFrame pour export (si tu veux)
    # ⚠️ Les noms de colonnes après OneHot + texte sont un peu longs à reconstruire,
    # mais tu peux déjà sauvegarder les matrices numpy ou utiliser directement ces
    # objets pour entraîner les modèles scikit-learn.

else:
    print("⚠️ train est vide, vérifie le chargement des données au début du notebook.")

Nombre de colonnes numériques avant filtrage: 25
Colonnes marquées à supprimer (corr > 0.95): 0
Colonnes supprimées pour forte corrélation (version finale):
[]
Shape X_final_train avant preprocessing : (160000, 26)
Shape X_train_preprocessed : (160000, 48)
Shape X_final_test avant preprocessing : (40000, 26)
Shape X_test_preprocessed : (40000, 48)
X_train: (128000, 48) X_valid: (32000, 48)


### 9. Export des données preprocessées en CSV

Cette section finale :
1. **Récupère les identifiants** (`customer_id`) depuis les données brutes
2. **Reconstruit les DataFrames** avec les noms de colonnes après preprocessing
3. **Réintègre la cible** (`target_is_fraud`) pour le train
4. **Exporte** `train_preprocessed_v1.csv` et `test_preprocessed_v1.csv` prêts pour l'entraînement des modèles

Les fichiers exportés contiennent :
- **Train** : toutes les features preprocessées + `customer_id` + `target_is_fraud`
- **Test** : toutes les features preprocessées + `customer_id`

In [22]:
# Export des données preprocessées en CSV

if not train.empty and 'X_train_preprocessed' in locals() and 'X_test_preprocessed' in locals():
    
    # 1) Récupérer les noms de colonnes après preprocessing
    # (après OneHotEncoder, les colonnes catégorielles sont transformées en plusieurs colonnes)
    try:
        # Récupérer les noms de colonnes depuis le ColumnTransformer
        feature_names = []
        
        # Colonnes numériques (gardent leur nom)
        for col in num_cols_final:
            feature_names.append(col)
        
        # Colonnes catégorielles (OneHotEncoder crée plusieurs colonnes)
        if len(cat_cols_final) > 0:
            # On doit récupérer les noms depuis le OneHotEncoder
            cat_encoder = categorical_transformer_final.named_steps['onehot']
            cat_feature_names = cat_encoder.get_feature_names_out(cat_cols_final)
            feature_names.extend(cat_feature_names)
        
        # Colonnes texte (longueur du texte)
        for col in text_cols_final:
            feature_names.append(f"{col}_length")
        
        # Filtrer les colonnes supprimées par VarianceThreshold
        # VarianceThreshold supprime les colonnes constantes, donc on doit vérifier
        var_thresh = full_pipeline_final.named_steps['var_thresh']
        selected_features = var_thresh.get_support()
        feature_names_final = [name for i, name in enumerate(feature_names) if selected_features[i]]
        
    except Exception as e:
        print(f"⚠️ Erreur lors de la récupération des noms de colonnes : {e}")
        print("Utilisation de noms génériques (feature_0, feature_1, ...)")
        feature_names_final = [f"feature_{i}" for i in range(X_train_preprocessed.shape[1])]
    
    # 2) Créer les DataFrames avec les noms de colonnes
    df_train_export = pd.DataFrame(X_train_preprocessed, columns=feature_names_final)
    df_test_export = pd.DataFrame(X_test_preprocessed, columns=feature_names_final)
    
    # 3) Réintégrer customer_id et target_is_fraud
    # Utiliser les identifiants sauvegardés avant le preprocessing
    if customer_ids_train is not None:
        df_train_export[ID_COL] = customer_ids_train.values
    elif ID_COL in train.columns:
        df_train_export[ID_COL] = train[ID_COL].values
    
    if customer_ids_test is not None:
        df_test_export[ID_COL] = customer_ids_test.values
    elif ID_COL in test.columns:
        df_test_export[ID_COL] = test[ID_COL].values
    
    if y_final is not None:
        df_train_export[TARGET_COL] = y_final.values
    
    # Réorganiser les colonnes : customer_id et target en premier
    cols_order = [ID_COL]
    if TARGET_COL in df_train_export.columns:
        cols_order.append(TARGET_COL)
    cols_order.extend([col for col in df_train_export.columns if col not in cols_order])
    
    df_train_export = df_train_export[cols_order]
    df_test_export = df_test_export[[ID_COL] + [col for col in df_test_export.columns if col != ID_COL]]
    
    # 4) Export en CSV
    OUTPUT_DIR = "../data"
    import os
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    train_export_path = os.path.join(OUTPUT_DIR, "train_preprocessed_v1_V4.csv")
    test_export_path = os.path.join(OUTPUT_DIR, "test_preprocessed_v1_V4.csv")
    
    df_train_export.to_csv(train_export_path, index=False)
    df_test_export.to_csv(test_export_path, index=False)
    
    print("✅ Export terminé avec succès !")
    print(f"📁 Train preprocessé : {train_export_path}")
    print(f"   Shape : {df_train_export.shape}")
    print(f"   Colonnes : {df_train_export.shape[1]} features + {ID_COL} + {TARGET_COL}")
    print(f"\n📁 Test preprocessé : {test_export_path}")
    print(f"   Shape : {df_test_export.shape}")
    print(f"   Colonnes : {df_test_export.shape[1]} features + {ID_COL}")
    
    print("\n📊 Aperçu du train exporté :")
    display(df_train_export.head())
    
    print("\n📊 Aperçu du test exporté :")
    display(df_test_export.head())
    
else:
    print("⚠️ Impossible d'exporter : les données preprocessées ne sont pas disponibles.")
    print("Assure-toi d'avoir exécuté toutes les cellules précédentes dans l'ordre.")

⚠️ Erreur lors de la récupération des noms de colonnes : This OneHotEncoder instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.
Utilisation de noms génériques (feature_0, feature_1, ...)
✅ Export terminé avec succès !
📁 Train preprocessé : ../data\train_preprocessed_v1_V4.csv
   Shape : (160000, 50)
   Colonnes : 50 features + customer_id + target_is_fraud

📁 Test preprocessé : ../data\test_preprocessed_v1_V4.csv
   Shape : (40000, 49)
   Colonnes : 49 features + customer_id

📊 Aperçu du train exporté :


,customer_id,target_is_fraud,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47
0,cust_6o9q8d4i36,0,-0.366226,5.071000,0.066844,-2.102801,-0.427195,-0.016780,-0.853861,-0.591332,0.225661,1.102521,-0.942105,-0.355949,-0.295522,-0.645078,-0.470355,-1.007851,2.786004,-0.357561,0.732273,-0.904644,0.389642,-0.211156,1.786369,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,cust_fgugtw230c,0,0.846856,-0.897542,-0.782516,0.549070,-0.213768,1.210934,1.064029,-0.973450,-0.891403,-0.592150,-1.185147,1.702631,-0.295522,-0.645078,-0.470355,0.489178,-0.657118,0.499671,0.231100,-1.163429,-0.825825,-0.778617,-1.258967,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,cust_8zi3lcbz0w,1,-0.972766,-1.010156,-0.763739,0.832604,0.639939,0.287570,0.757181,-0.616253,0.225661,-0.592150,-0.488159,1.019438,-0.295522,-0.645078,-0.470355,0.471690,0.691314,-1.308270,-0.914740,0.926518,-0.631841,0.890256,-0.347128,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,cust_5mp3ar41cj,0,0.586910,1.748887,0.058562,0.549070,-1.067476,-0.337918,-0.369388,1.103276,0.225661,1.102521,1.299504,0.684344,-0.295522,-0.645078,-0.470355,-1.130113,-1.960991,-1.687273,0.852980,1.285827,1.064968,0.991244,-0.056767,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,cust_gnpl83jb0j,0,-0.106280,1.579966,-0.639493,-1.268879,-1.921183,0.919295,1.700656,0.064913,-0.891403,-0.592150,0.314281,-1.931394,-0.295522,-0.645078,-0.470355,-0.874555,0.325311,-1.519629,-0.272579,-0.609482,1.329661,0.788669,-0.970339,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0



📊 Aperçu du test exporté :


,customer_id,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,feature_11,feature_12,feature_13,feature_14,feature_15,feature_16,feature_17,feature_18,feature_19,feature_20,feature_21,feature_22,feature_23,feature_24,feature_25,feature_26,feature_27,feature_28,feature_29,feature_30,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40,feature_41,feature_42,feature_43,feature_44,feature_45,feature_46,feature_47
0,cust_e5rx1bc9ii,0.413613,1.974115,-0.525078,-0.518350,0.426512,1.079025,1.886055,-0.391966,-0.891403,-0.592150,-1.184143,-1.943397,-0.295522,-0.645078,2.126052,-1.164497,0.516803,-0.266681,0.504709,-0.692611,-0.274105,0.254468,-0.792623,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,cust_bhwiukergn,-1.406010,-0.728621,0.372770,0.482356,-0.213768,-0.178668,-0.178873,-0.649480,0.225661,-0.592150,-1.013411,0.432273,-0.295522,-0.645078,-0.470355,0.084758,0.308952,-0.016605,0.532996,0.283053,-0.985216,-2.218769,0.375330,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,cust_ext9na4chu,0.326964,-0.897542,-0.023496,1.583133,-1.067476,-0.412987,-0.234187,-0.416887,0.225661,-0.592150,0.931928,-0.105879,-0.295522,0.483865,-0.470355,0.786302,2.476147,-2.005825,-0.261767,-0.308321,0.801417,0.432016,-0.560593,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,cust_9fsje5r1ny,0.067018,0.115984,-0.312047,-1.619127,1.493647,-0.559286,0.088677,1.352483,-0.891403,1.102521,-1.062622,-0.780069,-0.295522,-0.645078,-0.470355,-0.115306,0.638336,0.118559,0.029568,-1.049191,-1.273436,2.828623,-0.075042,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,cust_gdqxmodbed,1.366748,-0.447086,-0.084717,-0.768526,-0.213768,0.223295,0.944337,0.804227,-0.891403,-0.592150,1.017294,-1.401244,-0.295522,-0.645078,-0.470355,-0.164697,2.003868,1.017806,-0.807952,0.213108,-0.275609,0.596212,0.719335,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
